# Translation Differences - *Der Fremde*

Compares the dialogue of two English translations of the same German novel:

| Key | Translation |
|---|---|
| **P** | Published translation |
| **U** | Unpublished translation |

## Part 1 — Dialogues
1. Load and normalize both dialogue files
2. Character crosswalk: map each translation's character names to shared IDs
3. Automatic alignment of the dialogue lines (1:1, splits/merges, lines found in only one translation)
4. Review of uncertain alignments
5. Per-line differences, aggregations, export
6. Visualizer

## 1. Configuration
All paths, labels and alignment parameters live here and nowhere else.

In [ ]:
import json
import os
import re
import tempfile
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path('.')
OUT_DIR  = DATA_DIR / 'translation_differences'
OUT_DIR.mkdir(exist_ok=True)

TRANSLATIONS = {
    'published': {
        'prefix': 'P',
        'label':  'Published translation',
        'file':   DATA_DIR / 'dialogue_the_mysterious_stranger.json',
    },
    'unpublished': {
        'prefix': 'U',
        'label':  'Unpublished translation',
        'file':   DATA_DIR / 'dialogue_the_stranger.json',
    },
}
SIDES = list(TRANSLATIONS)

CROSSWALK_FILE       = OUT_DIR / 'characters_crosswalk.json'
ALIGNMENT_DRAFT_FILE = OUT_DIR / 'alignment_draft.json'

EMBED_MODEL = 'BAAI/bge-base-en-v1.5'
CHUNK_WORDS = 180

CALIBRATE_SIM         = True
UNRELATED_MIN_POS_GAP = 0.30

MAX_GROUP       = 2
W_TEXT          = 0.70
W_SPEAKER       = 0.20
W_LEN           = 0.10
W_POS           = 0.50
MATCH_THRESHOLD = 0.55
GROUP_PENALTY   = 0.05
POS_BAND        = 0.10
GAP_PENALTY     = 0.08
HIGH_SIM        = 0.70
MEDIUM_SIM      = 0.45
MASK_NAMES      = True

def write_json_atomic(path, data):
    path = Path(path)
    fd, tmp = tempfile.mkstemp(dir=path.parent, suffix='.tmp')
    with os.fdopen(fd, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

print('Config loaded')
for side, t in TRANSLATIONS.items():
    print(f"  {t['label']:<25} {t['file'].name}   exists={t['file'].exists()}")

## 2. Load and normalize
- uses `current_text` (the validated text)
- splits composite speakers/partners ("X, Y and Z") into lists
- gives each line a stable ID (`P12`, `U12`) and a relative position in the story (0–1)

In [ ]:
NAME_SPLIT = re.compile(r'\s*,\s*(?:and\s+)?|\s+and\s+')

def split_names(value):
    if not value:
        return []
    if isinstance(value, list):
        parts = []
        for v in value:
            parts.extend(split_names(v))
        return parts
    return [p.strip() for p in NAME_SPLIT.split(value) if p and p.strip()]

def clean_text(t):
    t = (t or '').replace('\u00a0', ' ')
    return re.sub(r'\s+', ' ', t).strip()

def load_translation(side):
    cfg = TRANSLATIONS[side]
    with open(cfg['file'], encoding='utf-8') as f:
        data = json.load(f)
    raw = data['dialogues'] if isinstance(data, dict) else data
    raw = [d for d in raw if d.get('status') != 'deleted']
    raw.sort(key=lambda d: d['id'])
    n = len(raw)

    lines = []
    for order, d in enumerate(raw):
        text = clean_text(d.get('current_text') or d.get('text') or d.get('original_text'))
        lines.append({
            'uid':          f"{cfg['prefix']}{d['id']}",
            'translation':  side,
            'src_id':       d['id'],
            'order':        order,
            'position':     round(order / (n - 1), 4) if n > 1 else 0.0,
            'text':         text,
            'word_count':   len(text.split()),
            'speakers_raw': split_names(d.get('speaker_name')),
            'partners_raw': split_names(d.get('dialogue_partner')),
            'silent_raw':   split_names(d.get('silent_participants', [])),
        })
    return lines

LINES = {side: load_translation(side) for side in SIDES}

for side in SIDES:
    L = LINES[side]
    print(f"{TRANSLATIONS[side]['label']:<25} {len(L):>4} lines   {sum(l['word_count'] for l in L):>6} words")

## 3. Character crosswalk
Maps each translation's names onto shared character IDs.

- On the first run it is created from the seed below and saved to `translation_differences/characters_crosswalk.json`.
- On later runs the **existing file is loaded**, so any manual edits are kept. New names that appear in the data are only *added*, with status `unmapped`.

In [ ]:
CROSSWALK_SEED = [
    {'id': 'franziska',  'names': {'published': ['Franziska von Fahnenberg'], 'unpublished': ['Franziska von Fahnenberg']}},
    {'id': 'franz',      'names': {'published': ['Franz von Kronstein'],      'unpublished': ['Franz von Kronstein']}},
    {'id': 'bertha',     'names': {'published': ['Bertha'],                   'unpublished': ['Bertha']}},
    {'id': 'fahnenberg', 'names': {'published': ['Knight of Fahnenberg'],     'unpublished': ['Baron von Fahnenberg']}},
    {'id': 'guide',      'names': {'published': ['Kumpan'],                   'unpublished': ['Tamek']}},
    {'id': 'azzo',       'names': {'published': ['Azzo'],                     'unpublished': ['Azzo von Klatka']}},
    {'id': 'woislaw',    'names': {'published': ['Woislaw'],                  'unpublished': ['Woislaw']}},
    {'id': 'steward',    'names': {'published': ['Steward', 'Chief Steward'], 'unpublished': ['Chief steward']}},
    {'id': 'servant',    'names': {'published': ['Servant'],                  'unpublished': ['Servant']}},
]

def slugify(name):
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')

def default_display(names):
    p = names.get('published') or []
    u = names.get('unpublished') or []
    if p and u and p[0].lower() == u[0].lower():
        return p[0]
    return ' / '.join(x for x in (p[:1] + u[:1]) if x)

def load_or_create_crosswalk():
    if CROSSWALK_FILE.exists():
        with open(CROSSWALK_FILE, encoding='utf-8') as f:
            cw = json.load(f)
        print(f"Loaded existing crosswalk ({len(cw['characters'])} characters) — manual edits are kept")
    else:
        cw = {'characters': [dict(c, display=default_display(c['names']), status='seed')
                            for c in CROSSWALK_SEED]}
        print('Created crosswalk from seed')
    return cw

def build_lookup(cw):
    lookup = {side: {} for side in SIDES}
    for c in cw['characters']:
        for side in SIDES:
            for nm in c['names'].get(side, []):
                lookup[side][nm.strip().lower()] = c['id']
    return lookup

crosswalk = load_or_create_crosswalk()
LOOKUP = build_lookup(crosswalk)

unmapped = {side: set() for side in SIDES}
for side in SIDES:
    for l in LINES[side]:
        for nm in l['speakers_raw'] + l['partners_raw'] + l['silent_raw']:
            if nm.lower() not in LOOKUP[side]:
                unmapped[side].add(nm)

added = 0
for side in SIDES:
    for nm in sorted(unmapped[side]):
        cid = slugify(nm)
        entry = next((c for c in crosswalk['characters'] if c['id'] == cid), None)
        if entry is None:
            entry = {'id': cid, 'display': nm, 'names': {s: [] for s in SIDES}, 'status': 'unmapped'}
            crosswalk['characters'].append(entry)
        entry['names'].setdefault(side, []).append(nm)
        added += 1

write_json_atomic(CROSSWALK_FILE, crosswalk)
LOOKUP = build_lookup(crosswalk)
CHARACTERS = {c['id']: c for c in crosswalk['characters']}

print(f"Saved → {CROSSWALK_FILE}")
if added:
    print(f"\n{added} name(s) were not in the crosswalk and were added as 'unmapped' — check them:")
    for side in SIDES:
        for nm in sorted(unmapped[side]):
            print(f"   [{side}] {nm}")
else:
    print('All names in both translations are mapped.')

print()
print(pd.DataFrame([{
    'id':          c['id'],
    'display':     c['display'],
    'published':   ', '.join(c['names'].get('published', [])),
    'unpublished': ', '.join(c['names'].get('unpublished', [])),
    'status':      c.get('status', ''),
} for c in crosswalk['characters']]).to_string(index=False))

## 4. Apply canonical character IDs
Every line gets `speakers`, `partners` and `silent` as canonical IDs. The cell also reports attribution warnings (a speaker also listed as partner or silent, or a line with no speaker).

In [ ]:
def to_canon(names, side):
    out = []
    for nm in names:
        cid = LOOKUP[side][nm.strip().lower()]
        if cid not in out:
            out.append(cid)
    return out

warnings = []
for side in SIDES:
    for l in LINES[side]:
        l['speakers'] = to_canon(l['speakers_raw'], side)
        l['partners'] = to_canon(l['partners_raw'], side)
        l['silent']   = to_canon(l['silent_raw'], side)

        overlap = set(l['speakers']) & (set(l['partners']) | set(l['silent']))
        if overlap:
            warnings.append(f"{l['uid']}: speaker also listed as partner/silent → {sorted(overlap)}")
        if not l['speakers']:
            warnings.append(f"{l['uid']}: no speaker")

LINE_BY_UID = {l['uid']: l for side in SIDES for l in LINES[side]}

print(f"Canonical IDs applied to {len(LINE_BY_UID)} lines")
print('\n'.join(warnings) if warnings else 'No attribution warnings')

## 5. First overview: speech per character in each translation
A line spoken by several characters together counts for each of them.

In [ ]:
rows = []
for cid, c in CHARACTERS.items():
    row = {'character': c['display']}
    for side in SIDES:
        own = [l for l in LINES[side] if cid in l['speakers']]
        row[f'{side}_lines'] = len(own)
        row[f'{side}_words'] = sum(l['word_count'] for l in own)
    rows.append(row)

overview = pd.DataFrame(rows).sort_values('published_words', ascending=False)
overview['words_ratio_U/P'] = (overview['unpublished_words']
                                / overview['published_words'].replace(0, np.nan)).round(2)
display(overview)

## 6. Sentence embeddings
Every span of 1 to `MAX_GROUP` consecutive lines is embedded, so splits and merges can be scored as a single unit. Speeches longer than `CHUNK_WORDS` are embedded in chunks and averaged, weighted by chunk length, so long monologues aren't truncated by the model.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBED_MODEL)


def build_name_masks():
    masks = []
    for c in crosswalk['characters']:
        names = {n for side in SIDES for n in c['names'].get(side, [])}
        if len({n.lower() for n in names}) > 1:
            token = c['id'].capitalize()
            for n in names:
                masks.append((re.compile(r'\b' + re.escape(n) + r'\b', re.IGNORECASE), token, len(n)))
    masks.sort(key=lambda m: -m[2])
    return masks


NAME_MASKS = build_name_masks() if MASK_NAMES else []


def mask_names(text):
    for pattern, token, _ in NAME_MASKS:
        text = pattern.sub(token, text)
    return text


print('Name masks:')
for pattern, token, _ in NAME_MASKS:
    print(f"   {pattern.pattern.replace(chr(92) + 'b', ''):<28} → {token}")


def embed_texts(texts):
    chunks, owners, weights = [], [], []
    for idx, t in enumerate(texts):
        words = t.split() or ['']
        for s in range(0, len(words), CHUNK_WORDS):
            piece = words[s:s + CHUNK_WORDS]
            chunks.append(' '.join(piece))
            owners.append(idx)
            weights.append(len(piece))

    vecs = model.encode(chunks, batch_size=32, normalize_embeddings=True, show_progress_bar=False)

    out = np.zeros((len(texts), vecs.shape[1]), dtype=np.float32)
    for v, o, w in zip(vecs, owners, weights):
        out[o] += v * w
    norms = np.linalg.norm(out, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return out / norms


SPAN_EMB, SPAN_INFO = {}, {}
for side in SIDES:
    L = LINES[side]
    SPAN_EMB[side], SPAN_INFO[side] = {}, {}
    for k in range(1, MAX_GROUP + 1):
        texts, infos = [], []
        for i in range(len(L) - k + 1):
            sub = L[i:i + k]
            texts.append(' '.join(mask_names(l['text']) for l in sub))
            infos.append({
                'speakers': set().union(*[set(l['speakers']) for l in sub]),
                'line_speakers': [set(l['speakers']) for l in sub],
                'words':    sum(l['word_count'] for l in sub),
                'position': sub[0]['position'],
            })
        print(f"Embedding {side}: {len(texts)} span(s) of {k} line(s)")
        SPAN_EMB[side][k]  = embed_texts(texts)
        SPAN_INFO[side][k] = infos

OPS = [(a, b) for a in range(1, MAX_GROUP + 1) for b in range(1, MAX_GROUP + 1) if a == 1 or b == 1]
SIM = {(a, b): SPAN_EMB['published'][a] @ SPAN_EMB['unpublished'][b].T for a, b in OPS}

pos_p = np.array([l['position'] for l in LINES['published']])
pos_u = np.array([l['position'] for l in LINES['unpublished']])
far   = np.abs(pos_p[:, None] - pos_u[None, :]) > UNRELATED_MIN_POS_GAP
raw11 = SIM[(1, 1)]

SIM_FLOOR = float(np.median(raw11[far]))
print(f"\nUnrelated pairs (story distance > {UNRELATED_MIN_POS_GAP}): "
        f"median={SIM_FLOOR:.3f}  p95={np.percentile(raw11[far], 95):.3f}")

if CALIBRATE_SIM:
    SIM = {k: np.clip((v - SIM_FLOOR) / (1 - SIM_FLOOR), 0, 1) for k, v in SIM.items()}
    print(f"Similarities rescaled: {SIM_FLOOR:.3f} → 0, 1.0 → 1")

print('Similarity matrices:', {k: v.shape for k, v in SIM.items()})

## 7. Alignment (dynamic programming)
A monotonic alignment, since the story order is the same in both translations. Possible operations:

| Type | Meaning |
|---|---|
| `1:1` | one published line ↔ one unpublished line |
| `1:2` / `2:1` | split / merge |
| `published_only` | line present only in the published translation |
| `unpublished_only` | line present only in the unpublished translation |

Score of a pair = `W_TEXT·similarity + W_SPEAKER·same speaker + W_LEN·length ratio − W_POS·story distance`.

A pair is matched only if it scores above `MATCH_THRESHOLD`. Each extra line in a split/merge costs `GROUP_PENALTY`. The algorithm picks the sequence of operations with the highest total score.

In [ ]:
def speaker_agreement(sp, su):
    if len(sp['line_speakers']) >= len(su['line_speakers']):
        many, other = sp['line_speakers'], su['speakers']
    else:
        many, other = su['line_speakers'], sp['speakers']
    return sum(1 for s in many if s & other) / len(many)


def pair_combined(i, a, j, b):
    sp = SPAN_INFO['published'][a][i]
    su = SPAN_INFO['unpublished'][b][j]
    text_sim = float(SIM[(a, b)][i, j])
    agree    = speaker_agreement(sp, su)
    lr       = min(sp['words'], su['words']) / max(sp['words'], su['words'], 1)
    dpos     = abs(sp['position'] - su['position'])
    combined = W_TEXT * text_sim + W_SPEAKER * agree + W_LEN * lr - W_POS * dpos
    return combined, text_sim, agree, lr, dpos


def align():
    P, U = LINES['published'], LINES['unpublished']
    n, m = len(P), len(U)
    NEG = float('-inf')
    best = np.full((n + 1, m + 1), NEG)
    best[0, 0] = 0.0
    back = {}

    for i in range(n + 1):
        for j in range(m + 1):
            cur = best[i, j]
            if cur == NEG:
                continue
            gap = cur - GAP_PENALTY
            if i < n and gap > best[i + 1, j]:
                best[i + 1, j] = gap
                back[(i + 1, j)] = (i, j, ('published_only',))
            if j < m and gap > best[i, j + 1]:
                best[i, j + 1] = gap
                back[(i, j + 1)] = (i, j, ('unpublished_only',))
            if i == n or j == m:
                continue
            if abs(P[i]['position'] - U[j]['position']) > POS_BAND:
                continue
            for a, b in OPS:
                if i + a > n or j + b > m:
                    continue
                gain = pair_combined(i, a, j, b)[0] - MATCH_THRESHOLD - GROUP_PENALTY * (a + b - 2)
                v = cur + gain
                if v > best[i + a, j + b]:
                    best[i + a, j + b] = v
                    back[(i + a, j + b)] = (i, j, ('match', a, b))

    steps, i, j = [], n, m
    while (i, j) != (0, 0):
        pi, pj, op = back[(i, j)]
        steps.append((pi, pj, op))
        i, j = pi, pj
    steps.reverse()
    return steps


def build_units(steps):
    P, U = LINES['published'], LINES['unpublished']
    units = []
    for pi, pj, op in steps:
        if op[0] == 'published_only':
            pub, unp, scores, utype = [P[pi]], [], None, 'published_only'
        elif op[0] == 'unpublished_only':
            pub, unp, scores, utype = [], [U[pj]], None, 'unpublished_only'
        else:
            a, b = op[1], op[2]
            pub, unp = P[pi:pi + a], U[pj:pj + b]
            combined, text_sim, agree, lr, dpos = pair_combined(pi, a, pj, b)
            scores = {
                'text_sim':          round(text_sim, 4),
                'speaker_agreement': round(agree, 3),
                'speaker_match':     agree == 1.0,
                'len_ratio':         round(lr, 3),
                'pos_delta':         round(dpos, 4),
                'combined':          round(combined, 4),
            }
            utype = f'{a}:{b}'

        if scores is None:
            confidence = 'n/a'
        elif scores['text_sim'] >= HIGH_SIM:
            confidence = 'high'
        elif scores['text_sim'] >= MEDIUM_SIM:
            confidence = 'medium'
        else:
            confidence = 'low'

        needs_review = (utype != '1:1') or confidence in ('low', 'n/a') \
                       or (scores is not None and not scores['speaker_match'])

        members = pub + unp
        units.append({
            'uid':          f'AL{len(units) + 1:03d}',
            'type':         utype,
            'published':    [l['uid'] for l in pub],
            'unpublished':  [l['uid'] for l in unp],
            'position':     round(float(np.mean([l['position'] for l in members])), 4),
            'scores':       scores,
            'confidence':   confidence,
            'needs_review': needs_review,
            'review':       {'status': 'pending' if needs_review else 'auto', 'note': ''},
        })
    return units


ALIGNMENT = build_units(align())

seen = [uid for a in ALIGNMENT for uid in a['published'] + a['unpublished']]
assert len(seen) == len(set(seen)) == len(LINE_BY_UID), 'alignment does not cover every line exactly once'

print(f"{len(ALIGNMENT)} alignment units")
for t, c in Counter(a['type'] for a in ALIGNMENT).most_common():
    print(f"   {t:<18} {c}")
print(f"\nConfidence:   {dict(Counter(a['confidence'] for a in ALIGNMENT))}")
print(f"Different speaker: {sum(1 for a in ALIGNMENT if a['scores'] and not a['scores']['speaker_match'])}")
print(f"Needs review: {sum(a['needs_review'] for a in ALIGNMENT)}")

## 8. Units to review
Shows every unit that is not a confident 1:1 match: splits/merges, lines found in only one translation, low confidence, or different speakers.

In [ ]:
def short(t, n=110):
    return t if len(t) <= n else t[:n - 1] + '…'


def show_unit(a):
    s = a['scores']
    head = f"{a['uid']}  [{a['type']}]  conf={a['confidence']}"
    if s:
        head += (f"  text={s['text_sim']:.2f}  speaker={'✓' if s['speaker_match'] else '✗'}"
                 f"  len={s['len_ratio']:.2f}")
    print(head)
    for side_key, mark in (('published', 'P'), ('unpublished', 'U')):
        for uid in a[side_key]:
            l = LINE_BY_UID[uid]
            who = ', '.join(CHARACTERS[c]['display'] for c in l['speakers'])
            print(f"   {mark} {uid:<6} {who:<24} {short(l['text'])}")
    print()


REVIEW = [a for a in ALIGNMENT if a['needs_review']]
print(f"{len(REVIEW)} units to review\n")
for a in REVIEW:
    show_unit(a)

## 9. Check against known cases
These are cases we already identified by hand. ✓ means the algorithm found them on its own.

In [ ]:
EXPECTED = {
    'unpublished_only': ['U64', 'U105', 'U106', 'U116'],
    'published_only':   ['P106', 'P115'],
    'groups': [
        (['P108'],        ['U108', 'U109']),
        (['P138'],        ['U139', 'U140']),
        (['P158'],        ['U159', 'U160']),
        (['P213'],        ['U215', 'U216']),
        (['P65', 'P66'],  ['U65']),
        (['P215'],        ['U218', 'U219']),
    ],
    'pairs': [
        ('P17', 'U16'), ('P53', 'U52'), ('P72', 'U71'), ('P150', 'U151'),
        ('P160', 'U162'), ('P174', 'U176'), ('P214', 'U217'), ('P217', 'U221'),
    ],
}


def unit_of(uid):
    return next(a for a in ALIGNMENT if uid in a['published'] + a['unpublished'])


def fmt(a):
    return f"{'+'.join(a['published']) or '—'} ↔ {'+'.join(a['unpublished']) or '—'} [{a['type']}]"


print('Lines expected in only one translation:')
for kind in ('unpublished_only', 'published_only'):
    for uid in EXPECTED[kind]:
        a = unit_of(uid)
        print(f"   {'✓' if a['type'] == kind else '✗'} {uid:<6} got: {fmt(a)}")

print('\nExpected splits / merges:')
for pub, unp in EXPECTED['groups']:
    a = unit_of(pub[0])
    ok = a['published'] == pub and a['unpublished'] == unp
    print(f"   {'✓' if ok else '✗'} {'+'.join(pub)} ↔ {'+'.join(unp):<12} got: {fmt(a)}")

print('\nExpected 1:1 pairs:')
for p, u in EXPECTED['pairs']:
    a = unit_of(p)
    ok = a['published'] == [p] and a['unpublished'] == [u]
    spk = '' if not a['scores'] else ('' if a['scores']['speaker_match'] else '  (different speaker)')
    print(f"   {'✓' if ok else '✗'} {p} ↔ {u:<6} got: {fmt(a)}{spk}")

checks = len(EXPECTED['unpublished_only']) + len(EXPECTED['published_only']) \
         + len(EXPECTED['groups']) + len(EXPECTED['pairs'])
print(f"\n{checks} checks")

## 10. Save the alignment draft
`translation_differences/alignment_draft.json` contains the parameters, characters, all lines, and the alignment. It can be regenerated at any time. Reviewed decisions will go into a separate file in the next stage, so this one never overwrites manual work.

In [ ]:
draft = {
    'meta': {
        'stage':        'alignment_draft',
        'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'translations': {
            side: {
                'label':       t['label'],
                'prefix':      t['prefix'],
                'source_file': t['file'].name,
                'lines':       len(LINES[side]),
                'words':       sum(l['word_count'] for l in LINES[side]),
            } for side, t in TRANSLATIONS.items()
        },
        'embedding_model': EMBED_MODEL,
        'params': {
            'max_group': MAX_GROUP, 'w_text': W_TEXT, 'w_speaker': W_SPEAKER, 'w_len': W_LEN,
            'w_pos': W_POS, 'match_threshold': MATCH_THRESHOLD, 'group_penalty': GROUP_PENALTY,
            'gap_penalty': GAP_PENALTY, 'pos_band': POS_BAND, 'high_sim': HIGH_SIM,
            'medium_sim': MEDIUM_SIM, 'mask_names': MASK_NAMES, 'chunk_words': CHUNK_WORDS,
            'calibrate_sim': CALIBRATE_SIM, 'sim_floor': round(SIM_FLOOR, 4),
            'unrelated_min_pos_gap': UNRELATED_MIN_POS_GAP,
        },
    },
    'characters': crosswalk['characters'],
    'lines':      LINE_BY_UID,
    'alignment':  ALIGNMENT,
}

write_json_atomic(ALIGNMENT_DRAFT_FILE, draft)
print(f"Saved → {ALIGNMENT_DRAFT_FILE}")
print(f"   {len(ALIGNMENT)} units | {sum(a['needs_review'] for a in ALIGNMENT)} to review")

## 11. Alignment review (interface)
Starts a local server at http://localhost:5002 for checking the alignment by hand.

- Decisions are saved to `translation_differences/alignment_reviewed.json`, created **once** from the draft.
- The draft can be regenerated without losing the review. If the draft changes structurally (different lines), the interface says so instead of overwriting anything.
- Re-running the cell stops the previous server and starts a new one, so code changes take effect without restarting the kernel.

In [ ]:
import threading
from flask import Flask, jsonify, request, send_file
from werkzeug.serving import make_server

REVIEWED_FILE = OUT_DIR / 'alignment_reviewed.json'
REVIEW_HTML   = Path('alignment_review.html')
REVIEW_PORT   = 5002


def validate_alignment(alignment, lines):
    seen = set()
    last = {side: -1 for side in SIDES}
    for u in alignment:
        if not u.get('published') and not u.get('unpublished'):
            return f"empty unit {u.get('uid')}"
        for side in SIDES:
            for uid in u.get(side, []):
                l = lines.get(uid)
                if l is None:
                    return f"unknown line {uid} in {u.get('uid')}"
                if l['translation'] != side:
                    return f"{uid} is on the wrong side in {u.get('uid')}"
                if uid in seen:
                    return f"{uid} appears twice"
                if l['order'] <= last[side]:
                    return f"story order broken at {uid} in {u.get('uid')}"
                last[side] = l['order']
                seen.add(uid)
    if len(seen) != len(lines):
        missing = sorted(set(lines) - seen)[:10]
        return f"{len(lines) - len(seen)} line(s) missing, e.g. {missing}"
    return None


def load_draft():
    with open(ALIGNMENT_DRAFT_FILE, encoding='utf-8') as f:
        return json.load(f)


def load_reviewed(draft):
    if REVIEWED_FILE.exists():
        with open(REVIEWED_FILE, encoding='utf-8') as f:
            rev = json.load(f)
        err = validate_alignment(rev['alignment'], draft['lines'])
        if err:
            raise ValueError(f"{REVIEWED_FILE.name} does not match the current draft ({err}). "
                             f"Rename or delete it to start the review again from the new draft.")
        return rev
    rev = {
        'meta': {
            'created_from_draft': draft['meta']['generated_at'],
            'created_at':         datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'last_saved':         None,
        },
        'alignment': draft['alignment'],
    }
    write_json_atomic(REVIEWED_FILE, rev)
    return rev


review_app = Flask('alignment_review')


@review_app.route('/')
def review_page():
    return send_file(REVIEW_HTML.resolve())


@review_app.route('/api/data')
def review_data():
    try:
        draft = load_draft()
        rev = load_reviewed(draft)
    except Exception as e:
        return jsonify({'error': str(e)}), 409
    return jsonify({
        'meta':        draft['meta'],
        'characters':  draft['characters'],
        'lines':       draft['lines'],
        'alignment':   rev['alignment'],
        'review_meta': rev['meta'],
    })


@review_app.route('/api/alignment', methods=['POST'])
def review_save():
    try:
        alignment = request.get_json(force=True)['alignment']
        draft = load_draft()
        err = validate_alignment(alignment, draft['lines'])
        if err:
            return jsonify({'error': err}), 400
        meta = {}
        if REVIEWED_FILE.exists():
            with open(REVIEWED_FILE, encoding='utf-8') as f:
                meta = json.load(f).get('meta', {})
        meta['last_saved'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        write_json_atomic(REVIEWED_FILE, {'meta': meta, 'alignment': alignment})
        return jsonify({'success': True, 'saved_at': meta['last_saved']})
    except Exception as e:
        return jsonify({'error': str(e)}), 500


if globals().get('REVIEW_SERVER') is not None:
    REVIEW_SERVER.shutdown()
    REVIEW_SERVER.server_close()
    print('Previous review server stopped')

REVIEW_SERVER = make_server('127.0.0.1', REVIEW_PORT, review_app, threaded=True)
threading.Thread(target=REVIEW_SERVER.serve_forever, daemon=True).start()

print(f"Review interface → http://localhost:{REVIEW_PORT}")
print(f"Decisions are saved to {REVIEWED_FILE}")

## 12. Review summary
Run after reviewing. Checks the structure of the reviewed file and lists what is still pending and what you changed by hand.

In [ ]:
with open(REVIEWED_FILE, encoding='utf-8') as f:
    reviewed = json.load(f)
RA = reviewed['alignment']

err = validate_alignment(RA, LINE_BY_UID)
print('Structure:', 'OK' if err is None else f'PROBLEM — {err}')
print(f"Last saved: {reviewed['meta'].get('last_saved')}\n")

print(f"{len(RA)} units")
for t, c in Counter(u['type'] for u in RA).most_common():
    print(f"   {t:<18} {c}")

print('\nReview status:', dict(Counter(u['review']['status'] for u in RA)))

pending = [u for u in RA if u['needs_review'] and u['review']['status'] == 'pending']
print(f"Pending: {len(pending)}" + (' → ' + ', '.join(u['uid'] for u in pending) if pending else ' — review complete'))

edited = [u for u in RA if u['review']['status'] == 'edited']
if edited:
    print('\nChanged by hand:')
    for u in edited:
        print(f"   {u['uid']:<8} {'+'.join(u['published']) or '—'} ↔ {'+'.join(u['unpublished']) or '—'} [{u['type']}]")

noted = [u for u in RA if u['review'].get('note')]
if noted:
    print('\nNotes:')
    for u in noted:
        print(f"   {u['uid']:<8} {u['review']['note']}")

# Phase 4 — Differences within each unit

Input: the reviewed alignment (`alignment_reviewed.json`). For every unit:

| Group | What is measured |
|---|---|
| Text | semantic similarity, lexical overlap, length, word-level diff, sentences with no counterpart |
| Attribution | speaker, addressee and silent participants in each translation |
| Lexical flags | forms of address, religious and supernatural vocabulary, archaisms, Germanisms, proper nouns, tone |
| Sentiment | polarity (negative / neutral / positive) and emotions (anger, disgust, fear, joy, neutral, sadness, surprise) for every line, then the shift per unit |

Every unit gets a **structure** (1:1, more lines in one translation, present in one only) and a **category** (near-identical, paraphrase, more content in one translation, divergent).

## 13. Configuration, word lists and helpers
`DIFF_PARAMS` holds every threshold. `FLAG_LEXICONS` holds the word lists behind the lexical flags: edit them freely (longer phrases are matched first, so "noble sir" is not also counted as "sir").

| Parameter | Meaning |
|---|---|
| `near_identical_sim` | calibrated similarity at or above which a unit is *near-identical* |
| `divergent_sim` | below this, a matched unit is *divergent* |
| `sentence_match_sim` | a sentence with no counterpart above this counts as present in one translation only |
| `min_sentence_words` | shorter sentences (interjections) are never reported as missing |
| `extra_content_words`, `extra_content_share` | minimum words, and share of the side's words, for a *content only in…* flag |
| `length_flag_ratio` | one side at least this many times longer → length flag |
| `polarity_shift` | change in polarity score that counts as a shift |
| `emotion_shift_jsd` | Jensen–Shannon divergence between emotion profiles that counts as a shift |
| `mutual_match_min_sim` | mutual best counterparts are matched even with different vocabulary above this |
| `sentiment_min_words` | below this on either side, sentiment shifts are marked unreliable and not counted |

The polarity model labels negative / neutral / positive; the emotion model labels anger, disgust, fear, joy, neutral, sadness, surprise.

In [ ]:
import difflib
import hashlib
import random

UNIT_DIFF_FILE        = OUT_DIR / 'unit_differences.json'
SENTIMENT_CACHE_FILE  = OUT_DIR / 'sentiment_cache.json'
VALIDATION_SAMPLE_CSV = OUT_DIR / 'sentiment_validation_sample.csv'

POLARITY_MODEL = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
EMOTION_MODEL  = 'j-hartmann/emotion-english-distilroberta-base'

DIFF_PARAMS = {
    'near_identical_sim':  0.80,
    'divergent_sim':       0.30,
    'sentence_match_sim':  0.35,
    'min_sentence_words':  4,
    'extra_content_words': 8,
    'extra_content_share': 0.20,
    'length_flag_ratio':   1.30,
    'polarity_shift':      0.30,
    'emotion_shift_jsd':   0.15,
    'mutual_match_min_sim': 0.10,
    'sentiment_min_words':  6,
}

FLAG_LEXICONS = {
    'address': [
        'noble sir', 'sir knight', 'sir baron', 'stern baron', 'most stern baron', 'stern young master',
        'young master', 'noble young lady', 'my fair young lady', 'fair young lady', 'fair lady',
        'dear lady', 'young lady', 'your grace', 'my lord', 'my poor friend', 'my friend',
        'old fellow', 'old comrade', 'foolish boy', 'thoughtless boy', 'gentlemen', 'ladies',
        'sir', 'madam', 'lady', 'miss', 'baron', 'knight', 'cousin', 'uncle', 'master', 'boy', 'wretch',
    ],
    'religious': [
        'most high', 'mortal sin', 'deadly sin', 'heaven', 'heavens', 'god', 'almighty', 'creator',
        'providence', 'lord', 'ave', 'credo', 'prayer', 'pray', 'praised', 'blessed', 'church',
        'sin', 'excommunication', 'hell', 'hellish', 'infernal', 'devil', 'christian',
    ],
    'supernatural': [
        'evil one', 'vampire', 'vampires', 'ghost', 'ghosts', 'spirit', 'phantom', 'apparition',
        'unearthly', 'monster', 'fiend', 'haunt', 'haunted', 'haunts', 'uncanny', 'blood', 'bloody',
        'coffin', 'grave', 'vault', 'crypt',
    ],
    'archaic': [
        'thou', 'thee', 'thy', 'thine', 'dost', 'doth', 'hath', 'ye', 'ere', 'whilst', 'wherefore',
        'hither', 'thither', 'yonder', 'darest', 'shalt', 'wilt',
    ],
    'germanisms': ['fräulein', 'ritter', 'herr', 'frau', 'junker', 'gnädige'],
}

STOPWORDS = {
    'a', 'an', 'the', 'and', 'or', 'but', 'if', 'so', 'of', 'to', 'in', 'on', 'at', 'by', 'for', 'with', 'from',
    'as', 'is', 'am', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did',
    'i', 'you', 'he', 'she', 'it', 'we', 'they', 'me', 'him', 'her', 'us', 'them', 'my', 'your', 'his', 'its',
    'our', 'their', 'this', 'that', 'these', 'those', 'not', 'no', 'yes', 'oh', 'ah', 'all', 'will', 'shall',
    'would', 'should', 'can', 'could', 'may', 'might', 'must', 'there', 'here', 'then', 'than', 'which', 'who',
    'what', 'when', 'where', 'how', 'why', 'very', 'too', 'also', 'only', 'now', 'even', 'more', 'most',
}

with open(REVIEWED_FILE, encoding='utf-8') as f:
    reviewed = json.load(f)
RA = reviewed['alignment']
err = validate_alignment(RA, LINE_BY_UID)
assert err is None, f'reviewed alignment is not valid: {err}'

WORD_RE    = re.compile(r"[^\W\d_]+(?:[’'\-][^\W\d_]+)*", re.UNICODE)
SENT_SPLIT = re.compile(r'(?<=[.!?…])\s+')
CAP_RE     = re.compile(r"[A-ZÀ-ÖØ-Þ][^\W\d_]*(?:[’'\-][^\W\d_]+)*")


def words(text):
    return WORD_RE.findall(text)


def split_sentences(text):
    parts = re.split(r'(?<=[.!?…])\s+|(?<=[.!?…])(?=[—–])', text)
    parts = [re.sub(r'^[—–\s]+', '', s).strip() for s in parts]
    parts = [s for s in parts if s]
    return parts or [text.strip()]


def find_terms(text, terms):
    t = ' ' + re.sub(r'\s+', ' ', text.lower()) + ' '
    found = Counter()
    for term in sorted(set(terms), key=len, reverse=True):
        pat = re.compile(r"(?<![\w’'])" + re.escape(term) + r"(?![\w’'])")
        n = len(pat.findall(t))
        if n:
            found[term] += n
            t = pat.sub(' ', t)
    return found


NAME_TOKENS = {tok for c in CHARACTERS.values() for side in SIDES
               for n in c['names'].get(side, []) for tok in n.split()}
PN_EXCLUDE = {w for terms in FLAG_LEXICONS.values() for t in terms for w in t.split()} | STOPWORDS


def proper_nouns(text):
    out = set()
    for m in CAP_RE.finditer(text):
        w = m.group(0)
        before = text[:m.start()].rstrip()
        if not before or before[-1] in '.!?—–-:;"“‘\'(…':
            continue
        if re.fullmatch(r"I(?:[’'][a-z]+)?", w) or w in NAME_TOKENS or w.lower() in PN_EXCLUDE:
            continue
        out.add(w)
    return out


def tone(text):
    return {
        'exclamations': text.count('!'),
        'questions':    text.count('?'),
        'dashes':       len(re.findall(r'[—–]', text)),
        'ellipses':     text.count('…') + text.count('...'),
    }


def content_words(text):
    return {w.lower() for w in words(text) if w.lower() not in STOPWORDS and len(w) > 2}


def jaccard(a, b):
    return len(a & b) / len(a | b) if (a or b) else 1.0


def unit_text(u, side):
    return ' '.join(LINE_BY_UID[uid]['text'] for uid in u[side])


def unit_words(u, side):
    return sum(LINE_BY_UID[uid]['word_count'] for uid in u[side])


def calibrate(sim):
    return np.clip((sim - SIM_FLOOR) / (1 - SIM_FLOOR), 0, 1)


print(f"{len(RA)} reviewed units loaded — structure OK")
print(f"Output → {UNIT_DIFF_FILE}")

## 14. Sentiment models + sanity check
Same principle as the embedding probe: before scoring the novel, the models must give sensible answers on sentences whose sentiment is unambiguous. If the cell prints ⚠, stop and check the installation before going further.

In [ ]:
from transformers import pipeline

polarity_clf = pipeline('text-classification', model=POLARITY_MODEL, top_k=None, truncation=True)
emotion_clf  = pipeline('text-classification', model=EMOTION_MODEL,  top_k=None, truncation=True)


def classify(clf, texts, batch_size=16):
    out = clf(list(texts), batch_size=batch_size)
    res = []
    for item in out:
        if isinstance(item, dict):
            item = [item]
        res.append({d['label'].lower(): float(d['score']) for d in item})
    return res


SENTIMENT_PROBE = [
    ("Heaven be praised that you live!",                                 {'positive'},             {'joy', 'surprise'}),
    ("I am dishonoured, dishonoured for ever!",                          {'negative'},             {'sadness', 'anger', 'disgust'}),
    ("No; it is blood! See, my hand is full of blood!",                  {'negative'},             {'fear', 'disgust', 'surprise'}),
    ("I detest him from the bottom of my heart.",                        {'negative'},             {'anger', 'disgust'}),
    ("Do you live in these ruins?",                                      {'neutral'},              {'neutral', 'surprise'}),
    ("I think we shall agree very well, and that it will be very nice.", {'positive'},             {'joy', 'neutral'}),
    ("They are worst when the wind rages; they become so mad and fierce.", {'negative', 'neutral'}, {'fear', 'anger'}),
]

pol = classify(polarity_clf, [s for s, _, _ in SENTIMENT_PROBE])
emo = classify(emotion_clf,  [s for s, _, _ in SENTIMENT_PROBE])

ok_pol = ok_emo = 0
print(f"{'':2} {'polarity':<22} {'emotion':<22} sentence")
for (text, exp_p, exp_e), p, e in zip(SENTIMENT_PROBE, pol, emo):
    lp, le = max(p, key=p.get), max(e, key=e.get)
    good_p, good_e = lp in exp_p, le in exp_e
    ok_pol += good_p
    ok_emo += good_e
    print(f"{'✓' if good_p and good_e else '✗'}  {lp:<9} {p[lp]:.2f}{'':7} {le:<9} {e[le]:.2f}{'':7} {text}")

print(f"\nPolarity correct: {ok_pol}/{len(SENTIMENT_PROBE)}   Emotion correct: {ok_emo}/{len(SENTIMENT_PROBE)}")
if ok_pol < len(SENTIMENT_PROBE) - 1 or ok_emo < len(SENTIMENT_PROBE) - 2:
    print("⚠ The models look unreliable in this environment — check the installation before going further.")

## 15. Sentiment for every line
Each line is scored **sentence by sentence**. The models read at most ~512 tokens, and long speeches mix moods. The sentence results are averaged, weighted by sentence length.

- `polarity_score` = P(positive) − P(negative), from −1 to +1.
- `emotion` is the dominant emotion. `emotion_nonneutral` is the strongest emotion other than neutral, which is more telling in dialogue, where "neutral" often wins.

Results are cached in `sentiment_cache.json`: re-running only scores lines whose text has changed.

In [ ]:
POLARITY_LABELS = ['negative', 'neutral', 'positive']
EMOTION_LABELS  = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']


def summarize_sentiment(pol, emo):
    non_neutral = {k: v for k, v in emo.items() if k != 'neutral'}
    return {
        'polarity':           {k: round(pol.get(k, 0.0), 4) for k in POLARITY_LABELS},
        'polarity_score':     round(pol.get('positive', 0.0) - pol.get('negative', 0.0), 4),
        'polarity_label':     max(POLARITY_LABELS, key=lambda k: pol.get(k, 0.0)),
        'emotions':           {k: round(emo.get(k, 0.0), 4) for k in EMOTION_LABELS},
        'emotion':            max(EMOTION_LABELS, key=lambda k: emo.get(k, 0.0)),
        'emotion_nonneutral': max(non_neutral, key=non_neutral.get) if non_neutral else None,
    }


def weighted_mix(parts):
    total = sum(w for _, w in parts) or 1
    mix = Counter()
    for dist, w in parts:
        for k, v in dist.items():
            mix[k] += v * w / total
    return dict(mix)


def cache_key(text):
    return hashlib.sha1(f"v2|{POLARITY_MODEL}|{EMOTION_MODEL}|{text}".encode('utf-8')).hexdigest()


cache = {}
if SENTIMENT_CACHE_FILE.exists():
    with open(SENTIMENT_CACHE_FILE, encoding='utf-8') as f:
        cache = json.load(f)

todo = [l for l in LINE_BY_UID.values() if cache.get(l['uid'], {}).get('key') != cache_key(l['text'])]
print(f"{len(LINE_BY_UID) - len(todo)} lines from cache, {len(todo)} to score")

if todo:
    jobs = [(l['uid'], s, max(1, len(words(s)))) for l in todo for s in split_sentences(l['text'])]
    print(f"Scoring {len(jobs)} sentences…")
    pol_all = classify(polarity_clf, [s for _, s, _ in jobs])
    emo_all = classify(emotion_clf,  [s for _, s, _ in jobs])

    per_line = {}
    for (uid, s, w), p, e in zip(jobs, pol_all, emo_all):
        per_line.setdefault(uid, []).append((p, e, w))

    for l in todo:
        parts = per_line[l['uid']]
        pol = weighted_mix([(p, w) for p, _, w in parts])
        emo = weighted_mix([(e, w) for _, e, w in parts])
        res = summarize_sentiment(pol, emo)
        res['sentences'] = len(parts)
        cache[l['uid']] = {'key': cache_key(l['text']), 'result': res}

    write_json_atomic(SENTIMENT_CACHE_FILE, cache)

for uid, l in LINE_BY_UID.items():
    l['sentiment'] = cache[uid]['result']

rows = []
for side in SIDES:
    S = [l['sentiment'] for l in LINES[side]]
    row = {'translation': TRANSLATIONS[side]['label'],
           'mean polarity': round(float(np.mean([s['polarity_score'] for s in S])), 3)}
    row.update({f'% {k}': round(100 * sum(s['polarity_label'] == k for s in S) / len(S), 1) for k in POLARITY_LABELS})
    rows.append(row)
display(pd.DataFrame(rows))

emo_rows = []
for side in SIDES:
    S = [l['sentiment'] for l in LINES[side]]
    c = Counter(s['emotion_nonneutral'] for s in S)
    emo_rows.append({'translation': TRANSLATIONS[side]['label'],
                     **{k: c.get(k, 0) for k in EMOTION_LABELS if k != 'neutral'}})
print('Dominant non-neutral emotion per line (count of lines):')
display(pd.DataFrame(emo_rows))

## 16. Lexicon check (VADER)
A second, transparent polarity measure built from a word list. It is independent of the model, so agreement between the two increases confidence in the results; where they disagree, the model is reading context a word list cannot. Optional: skipped if `vaderSentiment` is not installed.

In [ ]:
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    vader = SentimentIntensityAnalyzer()
except ImportError:
    vader = None
    print('vaderSentiment is not installed — run  pip install vaderSentiment  to enable this check')

if vader:
    def vader_label(c):
        return 'positive' if c >= 0.05 else 'negative' if c <= -0.05 else 'neutral'

    print(f"{'translation':<25} {'agreement':>10} {'correlation':>12}")
    for side in SIDES:
        model_scores, lex_scores, agree = [], [], 0
        for l in LINES[side]:
            c = vader.polarity_scores(l['text'])['compound']
            l['sentiment']['lexicon_compound'] = round(c, 4)
            model_scores.append(l['sentiment']['polarity_score'])
            lex_scores.append(c)
            agree += vader_label(c) == l['sentiment']['polarity_label']
        r = float(np.corrcoef(model_scores, lex_scores)[0, 1])
        print(f"{TRANSLATIONS[side]['label']:<25} {agree / len(LINES[side]):>9.0%} {r:>12.2f}")
    print('\nAgreement = same positive/neutral/negative label as the model; correlation = between the two polarity scores.')

## 17. Differences per unit
- **Similarity** is recomputed for every unit (including the ones edited by hand), with the same model and calibration as the alignment.
- **Sentences with no counterpart:** inside each unit, every sentence is compared with every sentence of the other translation. A sentence with no match above `sentence_match_sim` counts as present in one translation only. This is how passages like the Duke Hans of Sagan anecdotes are found.
- **Word diff:** kept for display in the visualizer (shared vs translation-specific wording).
- **Category:** `near_identical` / `paraphrase` / `more_content_in_published` / `more_content_in_unpublished` / `divergent`, plus `published_only` / `unpublished_only`.

In [ ]:
def word_diff(p_text, u_text):
    a = re.findall(r'\S+\s*', p_text)
    b = re.findall(r'\S+\s*', u_text)
    key = lambda t: re.sub(r"[^\w’']", '', t.lower())
    sm = difflib.SequenceMatcher(None, [key(t) for t in a], [key(t) for t in b], autojunk=False)
    ops = {'equal': 'same', 'replace': 'changed', 'delete': 'published_only', 'insert': 'unpublished_only'}
    return [{'op': ops[tag], 'published': ''.join(a[i1:i2]), 'unpublished': ''.join(b[j1:j2])}
            for tag, i1, i2, j1, j2 in sm.get_opcodes()]


def jsd(p, q, keys):
    P = np.array([p.get(k, 0.0) for k in keys]) + 1e-12
    Q = np.array([q.get(k, 0.0) for k in keys]) + 1e-12
    P, Q = P / P.sum(), Q / Q.sum()
    M = (P + Q) / 2
    kl = lambda x, y: float(np.sum(x * np.log2(x / y)))
    return 0.5 * kl(P, M) + 0.5 * kl(Q, M)


def unit_sentiment(u, side):
    members = [LINE_BY_UID[uid] for uid in u[side]]
    if not members:
        return None
    pol = weighted_mix([(m['sentiment']['polarity'], m['word_count']) for m in members])
    emo = weighted_mix([(m['sentiment']['emotions'], m['word_count']) for m in members])
    return summarize_sentiment(pol, emo)


def side_people(u, side, field):
    out = []
    for uid in u[side]:
        for c in LINE_BY_UID[uid][field]:
            if c not in out:
                out.append(c)
    return out


both = [u for u in RA if u['published'] and u['unpublished']]
unit_vecs = {side: embed_texts([mask_names(unit_text(u, side)) for u in both]) for side in SIDES}
UNIT_SIM = {u['uid']: float(calibrate(unit_vecs['published'][k] @ unit_vecs['unpublished'][k]))
            for k, u in enumerate(both)}

UNIT_SENTS = {u['uid']: {side: split_sentences(unit_text(u, side)) if u[side] else [] for side in SIDES} for u in RA}
flat = [(u_uid, side, k, s) for u_uid, d in UNIT_SENTS.items() for side in SIDES for k, s in enumerate(d[side])]
sent_vecs = embed_texts([mask_names(s) for *_, s in flat])
SENT_VEC = {(u_uid, side, k): sent_vecs[n] for n, (u_uid, side, k, _) in enumerate(flat)}
print(f"Embedded {len(both)} units and {len(flat)} sentences")


def unmatched_sentences(u):
    sents = UNIT_SENTS[u['uid']]
    res = {side: [] for side in SIDES}
    if not (sents['published'] and sents['unpublished']):
        for side in SIDES:
            res[side] = [{'text': s, 'words': len(words(s)), 'best_sim': None} for s in sents[side]]
        return res
    Ep = np.stack([SENT_VEC[(u['uid'], 'published', k)] for k in range(len(sents['published']))])
    Eu = np.stack([SENT_VEC[(u['uid'], 'unpublished', k)] for k in range(len(sents['unpublished']))])
    S = calibrate(Ep @ Eu.T)
    best_u_for_p = S.argmax(axis=1)
    best_p_for_u = S.argmax(axis=0)
    mutual_p = {i for i, j in enumerate(best_u_for_p)
                if best_p_for_u[j] == i and S[i, j] >= DIFF_PARAMS['mutual_match_min_sim']}
    mutual_u = {best_u_for_p[i] for i in mutual_p}

    for side, best, mutual in (('published', S.max(axis=1), mutual_p), ('unpublished', S.max(axis=0), mutual_u)):
        for k, (s, b) in enumerate(zip(sents[side], best)):
            if k in mutual:
                continue
            if b < DIFF_PARAMS['sentence_match_sim'] and len(words(s)) >= DIFF_PARAMS['min_sentence_words']:
                res[side].append({'text': s, 'words': len(words(s)), 'best_sim': round(float(b), 3)})
    return res


UNITS = []
for u in RA:
    a, b = len(u['published']), len(u['unpublished'])
    if not a:
        structure = 'unpublished_only'
    elif not b:
        structure = 'published_only'
    elif a == b == 1:
        structure = '1:1'
    else:
        structure = 'more_lines_in_published' if a > b else 'more_lines_in_unpublished'

    texts = {side: unit_text(u, side) for side in SIDES}
    wc = {side: unit_words(u, side) for side in SIDES}
    unmatched = unmatched_sentences(u)
    sent = {side: unit_sentiment(u, side) for side in SIDES}

    rec = {
        'uid':         u['uid'],
        'structure':   structure,
        'type':        u['type'],
        'published':   u['published'],
        'unpublished': u['unpublished'],
        'position':    u['position'],
        'review':      u['review'],
        'manual':      u.get('confidence') == 'manual',
        'words':       wc,
        'sentiment':   sent,
        'unmatched_sentences': unmatched,
        'attribution': {},
        'lexicon':     {},
        'proper_nouns': {},
        'tone':        {side: tone(texts[side]) if u[side] else None for side in SIDES},
        'flags':       [],
    }
    for field in ('speakers', 'partners', 'silent'):
        p, q = side_people(u, 'published', field), side_people(u, 'unpublished', field)
        rec['attribution'][field] = {
            'published': p, 'unpublished': q,
            'published_only': sorted(set(p) - set(q)), 'unpublished_only': sorted(set(q) - set(p)),
        }
    for cat, terms in FLAG_LEXICONS.items():
        rec['lexicon'][cat] = {side: dict(find_terms(texts[side], terms)) if u[side] else {} for side in SIDES}
    pn = {side: proper_nouns(texts[side]) for side in SIDES}
    rec['proper_nouns'] = {
        'published': sorted(pn['published']), 'unpublished': sorted(pn['unpublished']),
        'published_only': sorted(pn['published'] - pn['unpublished']),
        'unpublished_only': sorted(pn['unpublished'] - pn['published']),
    }

    flags = []
    if structure in ('published_only', 'unpublished_only'):
        flags.append(structure)
        rec.update({'text_sim': None, 'lexical_overlap': None, 'len_ratio': None,
                    'diff': None, 'sentiment_delta': None, 'category': structure})
    else:
        if structure != '1:1':
            flags.append(structure)
        sim = UNIT_SIM[u['uid']]
        rec['text_sim'] = round(sim, 4)
        rec['lexical_overlap'] = round(jaccard(content_words(texts['published']), content_words(texts['unpublished'])), 4)
        rec['len_ratio'] = round(min(wc.values()) / max(max(wc.values()), 1), 3)
        rec['diff'] = word_diff(texts['published'], texts['unpublished'])

        for field, flag in (('speakers', 'speaker_diff'), ('partners', 'partner_diff'), ('silent', 'silent_diff')):
            at = rec['attribution'][field]
            if at['published_only'] or at['unpublished_only']:
                flags.append(flag)

        r = DIFF_PARAMS['length_flag_ratio']
        if wc['unpublished'] >= r * wc['published']:
            flags.append('longer_in_unpublished')
        elif wc['published'] >= r * wc['unpublished']:
            flags.append('longer_in_published')
        extra = {}
        for side in SIDES:
            n = sum(s['words'] for s in unmatched[side])
            extra[side] = n
            if n >= DIFF_PARAMS['extra_content_words'] and n >= DIFF_PARAMS['extra_content_share'] * wc[side]:
                flags.append(f'content_only_in_{side}')

        for cat in FLAG_LEXICONS:
            if rec['lexicon'][cat]['published'] != rec['lexicon'][cat]['unpublished']:
                flags.append(f'{cat}_diff')
        if rec['proper_nouns']['published_only'] or rec['proper_nouns']['unpublished_only']:
            flags.append('proper_noun_diff')
        tp, tu = rec['tone']['published'], rec['tone']['unpublished']
        if (tp['exclamations'] > 0) != (tu['exclamations'] > 0) or (tp['questions'] > 0) != (tu['questions'] > 0):
            flags.append('tone_diff')

        sp, su = sent['published'], sent['unpublished']
        delta = {
            'polarity_score': round(su['polarity_score'] - sp['polarity_score'], 4),
            'polarity_label_changed': sp['polarity_label'] != su['polarity_label'],
            'emotion_jsd': round(jsd(sp['emotions'], su['emotions'], EMOTION_LABELS), 4),
            'emotion_changed': sp['emotion_nonneutral'] != su['emotion_nonneutral'],
            'reliable': min(wc.values()) >= DIFF_PARAMS['sentiment_min_words'],
        }
        rec['sentiment_delta'] = delta
        if not delta['reliable']:
            flags.append('sentiment_short')
        else:
            if abs(delta['polarity_score']) >= DIFF_PARAMS['polarity_shift']:
                flags.append('polarity_shift')
            if delta['emotion_jsd'] >= DIFF_PARAMS['emotion_shift_jsd']:
                flags.append('emotion_shift')

        if sim >= DIFF_PARAMS['near_identical_sim'] and rec['len_ratio'] >= 0.8 and not any(extra.values()):
            category = 'near_identical'
        elif 'content_only_in_unpublished' in flags and 'content_only_in_published' not in flags:
            category = 'more_content_in_unpublished'
        elif 'content_only_in_published' in flags and 'content_only_in_unpublished' not in flags:
            category = 'more_content_in_published'
        elif sim < DIFF_PARAMS['divergent_sim']:
            category = 'divergent'
        else:
            category = 'paraphrase'
        rec['category'] = category

    rec['flags'] = flags
    UNITS.append(rec)

print(f"{len(UNITS)} units analysed\n")
print('Category:')
for k, v in Counter(r['category'] for r in UNITS).most_common():
    print(f"   {k:<30} {v}")
print('\nFlags (number of units):')
for k, v in Counter(f for r in UNITS for f in r['flags']).most_common():
    print(f"   {k:<30} {v}")

## 18. Reports
What the phase found, in readable form: length differences, passages present in one translation only, attribution differences, the largest sentiment shifts and the lexical inventory of each translation.

In [ ]:
def who(u, side):
    ids = side_people(u, side, 'speakers') if u.get(side) else []
    return ', '.join(CHARACTERS[c]['display'] for c in ids) or '—'


def ids(rec):
    return f"{'+'.join(rec['published']) or '—'} ↔ {'+'.join(rec['unpublished']) or '—'}"


BY_UID = {u['uid']: u for u in RA}

print('── Largest length differences ─────────────────────────────')
matched = [r for r in UNITS if r['len_ratio'] is not None]
for r in sorted(matched, key=lambda r: abs(r['words']['unpublished'] - r['words']['published']), reverse=True)[:10]:
    d = r['words']['unpublished'] - r['words']['published']
    print(f"   {ids(r):<16} P={r['words']['published']:>4}  U={r['words']['unpublished']:>4}  Δ={d:+5}  "
          f"{who(BY_UID[r['uid']], 'published')}")

print('\n── Content present in only one translation (inside matched units) ──')
for side in SIDES:
    rows = [(r, s) for r in matched for s in r['unmatched_sentences'][side]]
    print(f"\n   {TRANSLATIONS[side]['label']}: {len(rows)} sentence(s), {sum(s['words'] for _, s in rows)} words")
    for r, s in sorted(rows, key=lambda x: -x[1]['words'])[:8]:
        print(f"      {ids(r):<16} ({s['words']} w) {short(s['text'], 95)}")

print('\n── Attribution differences ─────────────────────────────────')
for r in UNITS:
    if 'speaker_diff' in r['flags'] or 'partner_diff' in r['flags']:
        sp, pa = r['attribution']['speakers'], r['attribution']['partners']
        fmt = lambda xs: ', '.join(CHARACTERS[c]['display'] for c in xs) or '—'
        print(f"   {ids(r):<16} speaker P: {fmt(sp['published']):<22} U: {fmt(sp['unpublished']):<22} "
              f"| addressee P: {fmt(pa['published']):<22} U: {fmt(pa['unpublished'])}")

print('\n── Largest sentiment shifts (polarity, unpublished − published) ──')
shifted = [r for r in matched if r['sentiment_delta'] and r['sentiment_delta']['reliable']]
for r in sorted(shifted, key=lambda r: abs(r['sentiment_delta']['polarity_score']), reverse=True)[:10]:
    sp, su, d = r['sentiment']['published'], r['sentiment']['unpublished'], r['sentiment_delta']
    print(f"   {ids(r):<16} {sp['polarity_label']:>8} {sp['polarity_score']:+.2f} → {su['polarity_label']:<8} "
          f"{su['polarity_score']:+.2f}  ({d['polarity_score']:+.2f})")

print('\n── Largest emotion shifts (Jensen–Shannon divergence) ─────')
for r in sorted(shifted, key=lambda r: r['sentiment_delta']['emotion_jsd'], reverse=True)[:10]:
    sp, su = r['sentiment']['published'], r['sentiment']['unpublished']
    print(f"   {ids(r):<16} {sp['emotion_nonneutral']:>9} → {su['emotion_nonneutral']:<9} "
          f"JSD={r['sentiment_delta']['emotion_jsd']:.2f}")

print('\n── Lexical flags: terms per translation (whole dialogue) ───')
for cat in FLAG_LEXICONS:
    tot = {side: Counter() for side in SIDES}
    for r in UNITS:
        for side in SIDES:
            tot[side].update(r['lexicon'][cat][side])
    print(f"\n   {cat}")
    for side in SIDES:
        top = ', '.join(f'{t} {n}' for t, n in tot[side].most_common(10)) or '—'
        print(f"      {TRANSLATIONS[side]['label']:<25} {sum(tot[side].values()):>4}  {top}")

## 19. Manual validation sample
30 randomly chosen units, both translations, with the model's labels. Fill in your own reading in the empty columns. This gives a measured accuracy instead of trusting the model blindly.

The file is created **once** and never overwritten, so your annotations are safe.

In [ ]:
SAMPLE_SIZE = 30

if VALIDATION_SAMPLE_CSV.exists():
    print(f"{VALIDATION_SAMPLE_CSV.name} already exists — not overwritten, so your annotations are safe.")
    print("Delete or rename it if you want a new sample.")
else:
    rng = random.Random(42)
    pool = [r for r in UNITS if r['len_ratio'] is not None]
    rows = []
    for r in rng.sample(pool, min(SAMPLE_SIZE, len(pool))):
        for side in SIDES:
            s = r['sentiment'][side]
            rows.append({
                'unit': r['uid'],
                'translation': TRANSLATIONS[side]['label'],
                'lines': '+'.join(r[side]),
                'speaker': who(BY_UID[r['uid']], side),
                'text': unit_text(BY_UID[r['uid']], side),
                'model_polarity': s['polarity_label'],
                'model_polarity_score': s['polarity_score'],
                'model_emotion': s['emotion'],
                'model_emotion_nonneutral': s['emotion_nonneutral'],
                'your_polarity': '',
                'your_emotion': '',
                'comment': '',
            })
    pd.DataFrame(rows).to_csv(VALIDATION_SAMPLE_CSV, index=False, encoding='utf-8-sig')
    print(f"Saved {len(rows)} rows ({len(rows) // 2} units × 2 translations) → {VALIDATION_SAMPLE_CSV}")
    print("Fill in 'your_polarity' (negative / neutral / positive) and 'your_emotion' "
          "(anger, disgust, fear, joy, neutral, sadness, surprise).")

## 20. Save
`translation_differences/unit_differences.json`: meta (models, thresholds, word lists), characters, all lines with their sentiment, and all units with their differences. Phase 5 builds on this file.

In [ ]:
unit_diff = {
    'meta': {
        'stage':          'unit_differences',
        'generated_at':   datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'translations':   {side: {'label': t['label'], 'prefix': t['prefix'], 'source_file': t['file'].name}
                           for side, t in TRANSLATIONS.items()},
        'reviewed_alignment_saved': reviewed['meta'].get('last_saved'),
        'embedding_model': EMBED_MODEL,
        'sim_floor':       round(SIM_FLOOR, 4),
        'polarity_model':  POLARITY_MODEL,
        'emotion_model':   EMOTION_MODEL,
        'lexicon_check':   'vaderSentiment' if globals().get('vader') else None,
        'params':          DIFF_PARAMS,
        'lexicons':        FLAG_LEXICONS,
        'labels':          {'polarity': POLARITY_LABELS, 'emotions': EMOTION_LABELS},
    },
    'characters': crosswalk['characters'],
    'lines':      LINE_BY_UID,
    'units':      UNITS,
}

write_json_atomic(UNIT_DIFF_FILE, unit_diff)
print(f"Saved → {UNIT_DIFF_FILE}")
print(f"   {len(UNITS)} units | {len(LINE_BY_UID)} lines with sentiment")

# Phase 5 — Aggregations and export

Turns the per-unit differences into comparisons you can cite, then exports the final file for the visualizer.

| Level | Question it answers |
|---|---|
| Global | Which translation is longer, what is present in only one, how the register differs (archaisms, religious vocabulary, tone), overall sentiment |
| Per character | How each character speaks in each translation: volume, sentiment, emotions, forms of address, attribution changes |
| Per relationship | How A speaks *to* B in each translation; which links exist in only one translation |
| Along the story | Where cuts/additions and sentiment differences are concentrated |

Needs only section 1 (reads `unit_differences.json`).

## 21. Load and helpers
`TIMELINE_BINS`: number of equal story segments. `CHAR_MIN_WORDS`: characters who speak fewer words than this get `reliability: low`, since their sentiment profile rests on too little text to compare.

In [ ]:
UNIT_DIFF_FILE = OUT_DIR / 'unit_differences.json'
FINAL_FILE     = OUT_DIR / 'translation_differences.json'

TIMELINE_BINS  = 20
CHAR_MIN_WORDS = 100

with open(UNIT_DIFF_FILE, encoding='utf-8') as f:
    UD = json.load(f)

SIDES      = list(UD['meta']['translations'])
SIDE_LABEL = {s: UD['meta']['translations'][s]['label'] for s in SIDES}
U_LINES    = UD['lines']
U_UNITS    = UD['units']
CHARS      = {c['id']: c for c in UD['characters']}
POL_LABELS = UD['meta']['labels']['polarity']
EMO_LABELS = UD['meta']['labels']['emotions']
SIDE_LINES = {s: sorted((l for l in U_LINES.values() if l['translation'] == s), key=lambda l: l['order'])
              for s in SIDES}
TOTAL_WORDS = {s: sum(l['word_count'] for l in SIDE_LINES[s]) for s in SIDES}


def r(x, n=4):
    return None if x is None else round(float(x), n)


def mix(parts):
    total = sum(w for _, w in parts)
    if not total:
        return None
    out = Counter()
    for d, w in parts:
        for k, v in d.items():
            out[k] += v * w / total
    return dict(out)


def jsd(p, q, keys):
    P = np.array([p.get(k, 0.0) for k in keys]) + 1e-12
    Q = np.array([q.get(k, 0.0) for k in keys]) + 1e-12
    P, Q = P / P.sum(), Q / Q.sum()
    M = (P + Q) / 2
    kl = lambda x, y: float(np.sum(x * np.log2(x / y)))
    return 0.5 * kl(P, M) + 0.5 * kl(Q, M)


def profile(lines):
    if not lines:
        return None
    words = sum(l['word_count'] for l in lines)
    pol = mix([(l['sentiment']['polarity'], l['word_count']) for l in lines])
    emo = mix([(l['sentiment']['emotions'], l['word_count']) for l in lines])
    non_neutral = {k: v for k, v in emo.items() if k != 'neutral'}
    labels = Counter(l['sentiment']['polarity_label'] for l in lines)
    return {
        'lines':              len(lines),
        'words':              words,
        'polarity_score':     r(pol.get('positive', 0) - pol.get('negative', 0)),
        'polarity':           {k: r(pol.get(k, 0)) for k in POL_LABELS},
        'polarity_labels':    {k: labels.get(k, 0) for k in POL_LABELS},
        'emotions':           {k: r(emo.get(k, 0)) for k in EMO_LABELS},
        'emotion_nonneutral': max(non_neutral, key=non_neutral.get) if non_neutral else None,
    }


def compare_profiles(p, q):
    if not (p and q):
        return None
    return {
        'words_delta':    q['words'] - p['words'],
        'polarity_delta': r(q['polarity_score'] - p['polarity_score']),
        'emotion_jsd':    r(jsd(p['emotions'], q['emotions'], EMO_LABELS)),
    }


def name(cid):
    return CHARS[cid]['display'] if cid in CHARS else cid


print(f"Loaded {len(U_UNITS)} units and {len(U_LINES)} lines from {UNIT_DIFF_FILE.name}")
for s in SIDES:
    print(f"   {SIDE_LABEL[s]:<25} {len(SIDE_LINES[s])} lines, {TOTAL_WORDS[s]} words")

## 22. Global comparison
Rates are given per 1000 words so the two translations are comparable despite their different lengths.

In [ ]:
ONLY = ('published_only', 'unpublished_only')
matched_units = [u for u in U_UNITS if u['structure'] not in ONLY]

summary = {'translations': {}, 'comparison': {}}
for s in SIDES:
    prof = profile(SIDE_LINES[s])
    w = TOTAL_WORDS[s]
    per_k = lambda n: r(1000 * n / w, 2)

    only_unit_words = sum(U_LINES[uid]['word_count'] for u in U_UNITS if u['structure'] == f'{s}_only' for uid in u[s])
    unmatched_words = sum(x['words'] for u in matched_units for x in u['unmatched_sentences'][s])

    tone_tot = Counter()
    for u in U_UNITS:
        if u['tone'][s]:
            tone_tot.update(u['tone'][s])
    lex_tot = {cat: Counter() for cat in UD['meta']['lexicons']}
    for u in U_UNITS:
        for cat in lex_tot:
            lex_tot[cat].update(u['lexicon'][cat][s])

    summary['translations'][s] = {
        'label':                SIDE_LABEL[s],
        'profile':              prof,
        'avg_words_per_line':   r(w / len(SIDE_LINES[s]), 2),
        'words_in_lines_only_here':     only_unit_words,
        'words_in_sentences_only_here': unmatched_words,
        'tone_per_1000_words':  {k: per_k(v) for k, v in tone_tot.items()},
        'lexicon_counts':       {cat: dict(c.most_common()) for cat, c in lex_tot.items()},
        'lexicon_per_1000_words': {cat: per_k(sum(c.values())) for cat, c in lex_tot.items()},
    }

sims = [u['text_sim'] for u in matched_units]
summary['comparison'] = {
    'units':            len(U_UNITS),
    'structure':        dict(Counter(u['structure'] for u in U_UNITS)),
    'categories':       dict(Counter(u['category'] for u in U_UNITS)),
    'flags':            dict(Counter(f for u in U_UNITS for f in u['flags']).most_common()),
    'mean_text_sim':    r(np.mean(sims)),
    'median_text_sim':  r(np.median(sims)),
    'near_identical_share': r(sum(u['category'] == 'near_identical' for u in matched_units) / len(matched_units)),
    'profiles':         compare_profiles(summary['translations']['published']['profile'],
                                         summary['translations']['unpublished']['profile']),
}

rows = []
for s in SIDES:
    t = summary['translations'][s]
    p = t['profile']
    rows.append({
        'translation':       t['label'],
        'lines':             p['lines'],
        'words':             p['words'],
        'words/line':        t['avg_words_per_line'],
        'only here: lines':  t['words_in_lines_only_here'],
        'only here: sentences': t['words_in_sentences_only_here'],
        'polarity':          p['polarity_score'],
        'top emotion':       p['emotion_nonneutral'],
        'archaic /1000w':    t['lexicon_per_1000_words'].get('archaic'),
        'religious /1000w':  t['lexicon_per_1000_words'].get('religious'),
        '! /1000w':          t['tone_per_1000_words'].get('exclamations'),
        '? /1000w':          t['tone_per_1000_words'].get('questions'),
    })
display(pd.DataFrame(rows).set_index('translation').T)

c = summary['comparison']
print(f"\nMatched units: mean similarity {c['mean_text_sim']}, median {c['median_text_sim']}, "
      f"near-identical {100 * c['near_identical_share']:.0f}%")
print('Emotion profile, word-weighted (published → unpublished):')
for k in EMO_LABELS:
    pp = summary['translations']['published']['profile']['emotions'][k]
    uu = summary['translations']['unpublished']['profile']['emotions'][k]
    print(f"   {k:<9} {pp:.3f} → {uu:.3f}  ({uu - pp:+.3f})")

## 23. Per character
For each character, in each translation: lines, words, share of the dialogue, sentiment, dominant emotion, how often they are addressed or silently present. The comparison adds similarity of their lines, content present in one translation only, attribution changes and the forms of address they use.

A line spoken by several characters together counts fully for each of them.

In [ ]:
def lines_with(cid, s, field):
    return [l for l in SIDE_LINES[s] if cid in l[field]]


per_character = {}
for cid, c in CHARS.items():
    rec = {'id': cid, 'display': c['display'], 'names': c['names'], 'sides': {}}
    for s in SIDES:
        spoken = lines_with(cid, s, 'speakers')
        prof = profile(spoken)
        rec['sides'][s] = {
            'profile':        prof,
            'share_of_words': r(prof['words'] / TOTAL_WORDS[s]) if prof else 0.0,
            'addressed_in':   len(lines_with(cid, s, 'partners')),
            'silent_in':      len(lines_with(cid, s, 'silent')),
            'reliability':    'none' if not prof else ('low' if prof['words'] < CHAR_MIN_WORDS else 'ok'),
        }

    units = [u for u in U_UNITS if any(cid in u['attribution']['speakers'][s] for s in SIDES)]
    matched = [u for u in units if u['text_sim'] is not None]
    address = {s: Counter() for s in SIDES}
    content_only = {s: 0 for s in SIDES}
    for u in units:
        for s in SIDES:
            if cid in u['attribution']['speakers'][s]:
                address[s].update(u['lexicon']['address'][s])
                content_only[s] += sum(x['words'] for x in u['unmatched_sentences'][s])

    rec['comparison'] = {
        'units':               len(units),
        'mean_text_sim':       r(np.mean([u['text_sim'] for u in matched])) if matched else None,
        'categories':          dict(Counter(u['category'] for u in units)),
        'flags':               dict(Counter(f for u in units for f in u['flags']).most_common()),
        'attribution_changes': [u['uid'] for u in units
                                if cid in u['attribution']['speakers']['published_only']
                                or cid in u['attribution']['speakers']['unpublished_only']],
        'content_only_words':  content_only,
        'address_used':        {s: dict(address[s].most_common()) for s in SIDES},
        'profiles':            compare_profiles(rec['sides']['published']['profile'],
                                                rec['sides']['unpublished']['profile']),
    }
    per_character[cid] = rec

rows = []
for cid, rec in per_character.items():
    P, U = rec['sides']['published'], rec['sides']['unpublished']
    if not (P['profile'] or U['profile']):
        continue
    cmp_ = rec['comparison']
    rows.append({
        'character':    rec['display'],
        'P lines':      P['profile']['lines'] if P['profile'] else 0,
        'U lines':      U['profile']['lines'] if U['profile'] else 0,
        'P words':      P['profile']['words'] if P['profile'] else 0,
        'U words':      U['profile']['words'] if U['profile'] else 0,
        'Δ words':      (U['profile']['words'] if U['profile'] else 0) - (P['profile']['words'] if P['profile'] else 0),
        'similarity':   cmp_['mean_text_sim'],
        'P polarity':   P['profile']['polarity_score'] if P['profile'] else None,
        'U polarity':   U['profile']['polarity_score'] if U['profile'] else None,
        'P emotion':    P['profile']['emotion_nonneutral'] if P['profile'] else None,
        'U emotion':    U['profile']['emotion_nonneutral'] if U['profile'] else None,
        'emotion JSD':  cmp_['profiles']['emotion_jsd'] if cmp_['profiles'] else None,
        'only in P (w)': cmp_['content_only_words']['published'],
        'only in U (w)': cmp_['content_only_words']['unpublished'],
        'attrib. diff': len(cmp_['attribution_changes']),
        'reliability':  min(P['reliability'], U['reliability'], key=['none', 'low', 'ok'].index),
    })
display(pd.DataFrame(rows).sort_values('P words', ascending=False).set_index('character'))

print('\nForms of address used by each character (published → unpublished):')
for cid, rec in per_character.items():
    a = rec['comparison']['address_used']
    if a['published'] or a['unpublished']:
        fmt = lambda d: ', '.join(f'{k} {v}' for k, v in d.items()) or '—'
        print(f"   {rec['display']:<28} {fmt(a['published'])}\n   {'':<28} → {fmt(a['unpublished'])}")

## 24. Relationships
**Relationships** are directed (A → B = lines A addresses to B), with a sentiment profile for each translation. Each relationship is marked as present in both translations or in only one.

In [ ]:
rel_lines = {}
for s in SIDES:
    for l in SIDE_LINES[s]:
        for a in l['speakers']:
            for b in l['partners']:
                if a != b:
                    rel_lines.setdefault((a, b), {x: [] for x in SIDES})[s].append(l)

relationships = []
for (a, b), d in rel_lines.items():
    prof = {s: profile(d[s]) for s in SIDES}
    status = 'both' if all(prof.values()) else ('published_only' if prof['published'] else 'unpublished_only')
    relationships.append({
        'source': a, 'target': b, 'status': status,
        'sides': prof,
        'comparison': compare_profiles(prof['published'], prof['unpublished']),
    })
relationships.sort(key=lambda x: -sum(p['words'] for p in x['sides'].values() if p))

## 25. Along the story
The dialogue is divided into `TIMELINE_BINS` equal segments by position in the story. For each segment: how similar the translations are, the words in each, the words present in one translation only, and the sentiment of each.

In [ ]:
edges = np.linspace(0, 1, TIMELINE_BINS + 1)
timeline = []
for k in range(TIMELINE_BINS):
    lo, hi = edges[k], edges[k + 1]
    units = [u for u in U_UNITS if lo <= u['position'] < hi or (k == TIMELINE_BINS - 1 and u['position'] == 1.0)]
    matched = [u for u in units if u['text_sim'] is not None]
    seg = {
        'bin': k, 'start': r(lo, 3), 'end': r(hi, 3),
        'units': [u['uid'] for u in units],
        'mean_text_sim': r(np.mean([u['text_sim'] for u in matched])) if matched else None,
        'flags': dict(Counter(f for u in units for f in u['flags'])),
        'sides': {},
    }
    for s in SIDES:
        lines = [U_LINES[uid] for u in units for uid in u[s]]
        prof = profile(lines)
        seg['sides'][s] = {
            'words':              prof['words'] if prof else 0,
            'words_only_here':    sum(x['words'] for u in units for x in u['unmatched_sentences'][s]),
            'polarity_score':     prof['polarity_score'] if prof else None,
            'emotions':           prof['emotions'] if prof else None,
            'emotion_nonneutral': prof['emotion_nonneutral'] if prof else None,
        }
    timeline.append(seg)

rows = []
for seg in timeline:
    P, U = seg['sides']['published'], seg['sides']['unpublished']
    rows.append({
        'segment':       f"{seg['start']:.2f}–{seg['end']:.2f}",
        'units':         len(seg['units']),
        'similarity':    seg['mean_text_sim'],
        'P words':       P['words'],
        'U words':       U['words'],
        'only in P':     P['words_only_here'],
        'only in U':     U['words_only_here'],
        'P polarity':    P['polarity_score'],
        'U polarity':    U['polarity_score'],
        'P emotion':     P['emotion_nonneutral'],
        'U emotion':     U['emotion_nonneutral'],
    })
display(pd.DataFrame(rows).set_index('segment'))

## 26. Final export
`translation_differences/translation_differences.json` is the single file the visualizer reads. It holds the meta (models, thresholds, word lists, aggregation rules), characters, lines, units, the global summary, per character, relationships and timeline.

In [ ]:
final = {
    'meta': {
        **UD['meta'],
        'stage':        'translation_differences',
        'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'unit_differences_generated_at': UD['meta']['generated_at'],
        'aggregation': {
            'timeline_bins':   TIMELINE_BINS,
            'char_min_words':  CHAR_MIN_WORDS,
            'sentiment_weighting': 'word-weighted average of line distributions',
            'multi_speaker_lines': 'a line spoken by several characters counts fully for each of them',
        },
    },
    'characters':    UD['characters'],
    'lines':         U_LINES,
    'units':         U_UNITS,
    'summary':       summary,
    'per_character': per_character,
    'relationships': relationships,
    'timeline':      timeline,
}

write_json_atomic(FINAL_FILE, final)
size_mb = FINAL_FILE.stat().st_size / 1e6
print(f"Saved → {FINAL_FILE}  ({size_mb:.1f} MB)")
print(f"   {len(U_UNITS)} units | {len(per_character)} characters | {len(relationships)} relationships | "
      f"{len(timeline)} timeline segments")

## 27. Sentiment validation
Compares the models' labels with the manual annotation of the 30-unit sample (60 lines). It reports accuracy and Cohen's kappa overall and for each translation, which tests whether the published translation's older register affects the models. It also checks whether the model sees the same direction of change between the translations that a human reader does.

In [ ]:
import csv

VALID_FILE = OUT_DIR / 'sentiment_validation_sample.csv'

def kappa(a, b):
    a, b = list(a), list(b)
    labels = sorted(set(a) | set(b))
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    ca, cb = Counter(a), Counter(b)
    pe = sum(ca[l] * cb[l] for l in labels) / n ** 2
    return (po - pe) / (1 - pe) if pe < 1 else 1.0

with open(VALID_FILE, encoding='utf-8-sig') as f:
    sep = csv.Sniffer().sniff(f.read(4096), delimiters=',;').delimiter
V = pd.read_csv(VALID_FILE, sep=sep, encoding='utf-8-sig')
V = V.loc[:, ~V.columns.str.startswith('Unnamed')]
for c in ['your_polarity', 'your_emotion', 'model_polarity', 'model_emotion', 'model_emotion_nonneutral']:
    V[c] = V[c].astype(str).str.strip().str.lower().replace({'nan': np.nan})
V = V.dropna(subset=['your_polarity', 'your_emotion'])
print(f"{len(V)} annotated rows ({V['unit'].nunique()} units)\n")

POL_ORDER = ['negative', 'neutral', 'positive']
pol_ok = V['model_polarity'] == V['your_polarity']
emo_ok = V['model_emotion'] == V['your_emotion']
emo_nn_ok = V['model_emotion_nonneutral'] == V['your_emotion']

def report(mask):
    sub = V[mask]
    return {
        'rows':                    len(sub),
        'polarity accuracy':       round(pol_ok[mask].mean(), 3),
        'polarity kappa':          round(kappa(sub['model_polarity'], sub['your_polarity']), 3),
        'polarity off by 2 (neg↔pos)': int(((sub['model_polarity'] == 'negative') & (sub['your_polarity'] == 'positive') |
                                            (sub['model_polarity'] == 'positive') & (sub['your_polarity'] == 'negative')).sum()),
        'emotion accuracy (top label)':        round(emo_ok[mask].mean(), 3),
        'emotion accuracy (top non-neutral)':  round(emo_nn_ok[mask].mean(), 3),
        'emotion kappa (top non-neutral)':     round(kappa(sub['model_emotion_nonneutral'], sub['your_emotion']), 3),
    }

rows = {'All': report(V.index == V.index)}
for t in V['translation'].unique():
    rows[t] = report(V['translation'] == t)
display(pd.DataFrame.from_dict(rows, orient='index'))

print('\nPolarity — rows: model, columns: you')
display(pd.crosstab(V['model_polarity'], V['your_polarity']).reindex(index=POL_ORDER, columns=POL_ORDER, fill_value=0))

print('Mean model polarity score by your label:')
print(V.groupby('your_polarity')['model_polarity_score'].agg(['mean', 'count']).reindex(POL_ORDER).round(3).to_string())

print('\nEmotion — rows: model (top non-neutral), columns: you')
display(pd.crosstab(V['model_emotion_nonneutral'], V['your_emotion']))

pairs = V.pivot_table(index='unit', columns='translation', values=['model_polarity_score', 'your_polarity'], aggfunc='first')
P_LAB, U_LAB = 'Published translation', 'Unpublished translation'
rank = {'negative': -1, 'neutral': 0, 'positive': 1}
d_model = pairs['model_polarity_score'][U_LAB] - pairs['model_polarity_score'][P_LAB]
d_you = pairs['your_polarity'][U_LAB].map(rank) - pairs['your_polarity'][P_LAB].map(rank)
changed = d_you != 0
print(f"\nUnits where YOUR polarity label differs between translations: {int(changed.sum())} of {len(pairs)}")
if changed.any():
    same_dir = (np.sign(d_model[changed]) == np.sign(d_you[changed])).mean()
    print(f"   model score moves in the same direction in {same_dir:.0%} of them")
big = d_model.abs() >= DIFF_PARAMS.get('polarity_shift', 0.3) if 'DIFF_PARAMS' in globals() else d_model.abs() >= 0.3
print(f"Units with a model polarity shift ≥ 0.3: {int(big.sum())}; you also see a change in {int((big & changed).sum())} of them")

print('\nPolarity disagreements:')
for _, row in V[~pol_ok].iterrows():
    print(f"   {row['unit']} {row['lines']:<9} model {row['model_polarity']:<8} ({row['model_polarity_score']:+.2f})  you {row['your_polarity']:<8} │ {str(row['text'])[:80]}")